In [3]:
# Chap 15.riss.kr 에서 특정 키워드로 논문 / 학술 자료 검색하기

#Step 1. 필요한 모듈을 로딩합니다
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
import time 

#Step 2. 사용자에게 검색 관련 정보들을 입력 받습니다.
print("=" *100)
print(" 이 크롤러는 RISS 사이트의 논문 및 학술자료 수집용 웹크롤러입니다.")
print("=" *100)
query_txt = input('1.수집할 자료의 키워드는 무엇입니까?: ')

#Step 3. 수집된 데이터를 저장할 파일 이름 입력받기 
f_dir = input("2.파일을 저장할 폴더명만 쓰세요(기본값:c:\\py_temp\\):")
if f_dir == '' :
    f_dir="c:\\py_temp\\"

#Step 4. 크롬 드라이버 설정 및 웹 페이지 열기
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

url = 'http://www.riss.kr/'
driver.get(url)
driver.maximize_window()
time.sleep(2)

#Step 5. 자동으로 검색어 입력 후 조회하기
element = driver.find_element(By.ID,'query')
element.send_keys(query_txt)
element.send_keys("\n")

#Step 6.학위 논문 선택하기
driver.find_element(By.LINK_TEXT,'학위논문').click()
time.sleep(2)

#Step 7.Beautiful Soup 로 본문 내용만 추출하기
from bs4 import BeautifulSoup
html_1 = driver.page_source
soup_1 = BeautifulSoup(html_1, 'html.parser')

#Step 8. 총 검색 건수를 보여주고 수집할 건수 입력받기
import math
total_cnt = soup_1.find('div','searchBox pd').find('span','num').get_text()
print('검색하신 키워드 %s (으)로 총 %s 건의 학위논문이 검색되었습니다' %(query_txt,total_cnt))
cnt = int(input('이 중에서 몇 건을 수집하시겠습니까?: '))
page_cnt = math.ceil(cnt / 10)
print('%s 건의 데이터를 수집하기 위해 %s 페이지의 게시물을 조회합니다.' %(cnt,page_cnt))
print("\n")

#Step 9. 데이터 수집하기
no2=[]           # 게시글 번호 컬럼
title2=[ ]       # 게시글 제목 컬럼
author2=[]       # 논문 저자 컬럼
company2=[ ]     # 소속 기관 컬럼
date2=[ ]        # 게시글 날짜 컬럼
suksa2=[ ]       # 국내석사 컬럼
contents2=[]     # 초록내용
full_url2=[]     # 논문 원본 URL

no = 1           # 게시글 번호 초기값
            
for a in range(1,page_cnt+1) :
    print("\n")
    print("%s 페이지 내용 수집 시작합니다 =======================" %a)

    time.sleep(2)
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    content_list = soup.find('div','srchResultListW').find_all('li')

    for i in content_list:
        # 논문 제목 체크하기
        try:
            title=i.find('p','title').get_text().strip()
        except :
            continue 
        else :
            # 1.게시글 번호
            print("\n")
            print("%s 번째 정보를 추출하고 있습니다============" %no)
            no2.append(no)
            print("1.번호 : %s" %no)
            
            # 2. 논문 제목
            title2.append(title.strip())
            print("2.제목 : %s" %title.strip())

            # 3. 작성자
            try :
                author=i.find('p','etc').find('span','writer').get_text().strip()
            except :
                author = '작성자가 없습니다'
                print("3.작성자 : %s" %author.strip())
                author2.append(author.strip())
            else :
                author2.append(author.strip())
                print("3.작성자 : %s" %author.strip())

            # 4. 소속기관
            try :
                company=i.find('p','etc').find('span','assigned').get_text().strip()
            except :
                company='소속 기관이 없습니다'
                company2.append(company.strip())
                print("4.소속기관 : %s" %company.strip())
            else :
                company2.append(company.strip())
                print("4.소속기관 : %s" %company.strip())

            # 5. 발표날짜
            try :
                date_1 =i.find('p','etc').find_all('span')
                date_2 = date_1[2].get_text().strip()
            except :
                date_2='발표날짜가 없습니다'
                date2.append(date_2)
                print("5.발표년도 : %s" %date_2)
            else :
                date2.append(date_2)
                print("5.발표년도 : %s" %date_2)

            # 6.학위여부
            try :
                suksa_1 =i.find('p','etc').find_all('span')
                suksa_2 = suksa_1[3].get_text().strip()
            except :
                suksa_2='학위가 없습니다'
                suksa2.append(suksa_2)
                print("6.학위여부 : %s" %suksa_2)
            else :
                suksa2.append(suksa_2)
                print("6.학위여부 : %s" %suksa_2)

            # 7.초록 내용-해당 논문의 상세 내역에서 추출할 수 있음.    
            url_1 = i.find('p','title').find('a')['href']
            full_url = 'http://www.riss.kr'+url_1
            time.sleep(1)
            driver.get(full_url)

            html_1 = driver.page_source
            soup_1 = BeautifulSoup(html_1, 'html.parser')  
            try :
                cont=soup_1.find("div","text").find('p').get_text().replace("\n","").strip()
            except :
                cont='초록이 없습니다'
                contents2.append(cont)
                print("7.초록내용 : %s" %cont)
            else :
                contents2.append(cont)
                print("7.초록내용 : %s" %cont)

            time.sleep(1)

            # 8.논문 url 주소
            full_url2.append(full_url)
            print('8.논문 URL 주소:' , full_url)

            driver.back()  # 이전 페이지로 돌아가기

            time.sleep(2)

            no += 1
            
            if no > cnt :
                break 
                            
    a += 1 
    b = str(a)

    try :
        driver.find_element(By.LINK_TEXT ,'%s' %b).click() 
    except :
        driver.find_element(By.LINK_TEXT,'다음 페이지로').click()
        
print("요청하신 작업이 모두 완료되었습니다")

# Step 10. 수집된 데이터를 xls와 csv 형태로 저장하기
# 현재 날짜와 시간으로 폴더 만들고 파일 이름 설정하기
import os

n = time.localtime()
s = '%04d-%02d-%02d-%02d-%02d-%02d' %(n.tm_year, n.tm_mon, n.tm_mday, n.tm_hour, n.tm_min, n.tm_sec)

os.makedirs(f_dir+'RISS'+'-'+s+'-'+'학위논문')

fc_name = f_dir+'RISS'+'-'+s+'-'+'학위논문'+'\\'+'RISS'+'-'+s+'-'+'학위논문'+'.csv'
fx_name = f_dir+'RISS'+'-'+s+'-'+'학위논문'+'\\'+'RISS'+'-'+s+'-'+'학위논문'+'.xls'

# 데이터 프레임 생성 후 xls , csv 형식으로 저장하기
import pandas as pd 

df = pd.DataFrame()
df['번호']=no2
df['제목']=pd.Series(title2)
df['저자']=pd.Series(author2)
df['소속(발행)기관']=pd.Series(company2)
df['날짜']=pd.Series(date2)
df['학위(논문일경우)']=pd.Series(suksa2)
df['초록(논문일경우)']=pd.Series(contents2)
df['자료URL주소']=pd.Series(full_url2)

# xls 형태로 저장하기
df.to_excel(fx_name,index=False, engine='openpyxl')

# csv 형태로 저장하기
df.to_csv(fc_name,index=False, encoding="utf-8-sig")

print('요청하신 데이터 수집 작업이 정상적으로 완료되었습니다')

 이 크롤러는 RISS 사이트의 논문 및 학술자료 수집용 웹크롤러입니다.
검색하신 키워드 메타버스 (으)로 총 1,327 건의 학위논문이 검색되었습니다
150 건의 데이터를 수집하기 위해 15 페이지의 게시물을 조회합니다.




1 페이지 내용 수집 시작합니다 =======================


1 번째 정보를 추출하고 있습니다============
1.번호 : 1
2.제목 : 소셜미디어(Social Media)  메타버스(Metaverse)를 활용한 설교 연구 : 미디어 변화를 중심으로
3.작성자 : 장용환
4.소속기관 : 호서대학교 연합신학전문대학원
5.발표년도 : 2022
6.학위여부 : 국내박사
7.초록내용 : 초록이 없습니다
8.논문 URL 주소: http://www.riss.kr/search/detail/DetailView.do?p_mat_type=be54d9b8bc7cdb09&control_no=30ffdea989b873c6ffe0bdc3ef48d419&keyword=메타버스


2 번째 정보를 추출하고 있습니다============
1.번호 : 2
2.제목 : 메타버스에서의 브랜드 체험이소비자브랜드 인게이지먼트 및 브랜드충성도에 미치는 영향 : 브랜드 진정성 조절효과
3.작성자 : 이광수
4.소속기관 : 경기대학교 대학원
5.발표년도 : 2024
6.학위여부 : 국내박사
7.초록내용 : 디지털 미디어 환경의 급격한 성장에 기업은 소비자들에게 브랜드의 가치나 메시지를 전달하는 것이 이전에 비해 어려움에 직면했다. 더불어 마케팅에서 소비자의 권한 강화는 차별화되고 ...
8.논문 URL 주소: http://www.riss.kr/search/detail/DetailView.do?p_mat_type=be54d9b8bc7cdb09&control_no=4dcf21c0520edcdeffe0bdc3ef48d419&keyword=메타버스


3 번째 정보를 추출하고 있습니다============
1.번호 : 3
2.제목 : 메타버스 분야 국가연구개발

In [ ]:
# Chap 15.riss.kr 에서 특정 키워드로 논문 / 학술 자료 검색하기

#Step 1. 필요한 모듈을 로딩합니다
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
import time
import math


#Step 2. 사용자에게 검색 관련 정보들을 입력 받습니다.
print("=" *60)
print("연습문제 : 서울시 응답소 게시판 크롤링하기 ")
print("=" *60)
cnt = int(input('1.크롤링 할 건수는 몇건입니까'))
page_cnt = math.ceil(cnt / 10)
print("크롤링 할 총 페이지 번호: %d" %page_cnt)
print("\n")

#Step 3. 수집된 데이터를 저장할 파일 이름 입력받기 
f_dir = input("2.파일을 저장할 폴더명만 쓰세요(기본값:c:\\py_temp\\):")
if f_dir == '' :
    f_dir="c:\\py_temp\\"

#Step 4. 크롬 드라이버 설정 및 웹 페이지 열기
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

url = 'https://eungdapso.seoul.go.kr/'
driver.get(url)
driver.maximize_window()
time.sleep(2)

#Step 5. 자동으로 검색어 입력 후 조회하기
driver.find_element(By.XPATH,'//*[@id="content"]/div[2]/a/div/div').click()
time.sleep(2)

#Step 7.Beautiful Soup 로 본문 내용만 추출하기
from bs4 import BeautifulSoup
html_1 = driver.page_source
soup_1 = BeautifulSoup(html_1, 'html.parser')

#Step 9. 데이터 수집하기
no2=[]           # 게시글 번호 컬럼
title2=[ ]       # 게시글 제목 컬럼
contents2=[]     # 공개일
contents3=[]     # 상담내용

no = 1           # 게시글 번호 초기값
cnt_page = 0
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys

for a in range(1,page_cnt+1) :
    print("\n")
    print("%s 페이지 내용 수집 시작합니다 =======================" %a)

    time.sleep(2)
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    content_list = soup.find('table','rp_tb').find_all('td')

    for i in content_list:
        
        try:
            title=i.find('div','rp_inlink_item').get_text().strip()
            title2.append(title)
        except :
            continue 
        else :
            # 1.게시글 번호
            print("\n")
            print("%s 번째 게시글의 상세내역을 추출하고 있습니다============" %no)
            no2.append(no)
            print("1.제목 : %s" %title.strip())

            link_element = driver.find_elements(By.CSS_SELECTOR, "a.rp_inlink")
            # JavaScript 실행하여 onView() 함수 호출
            driver.execute_script(link_element[cnt_page].get_attribute("href"))
            time.sleep(2)

            html_1 = driver.page_source
            soup_1 = BeautifulSoup(html_1, 'html.parser')  
            try :
                cont=soup_1.find("span","sclinkage_evalue").get_text().replace("\n","").strip()
            except :
                cont='공개일이 없습니다'
                contents2.append(cont)
                print("2.공개일 : %s" %cont)
            else :
                contents2.append(cont)
                print("2.공개일 : %s" %cont)
            
            try :
                cont2=soup_1.find("span","rp_indata").get_text().replace("\n","").strip()
            except :
                cont2='상담내용이 없습니다'
                contents3.append(cont2)
                print("3.상담내용 : %s" %cont2)
            else :
                contents3.append(cont2)
                print("3.상담내용 : %s" %cont2)
            time.sleep(1)

            driver.back()  # 이전 페이지로 돌아가기

            time.sleep(2)

            no += 1
            cnt_page += 1
            if no > cnt :
                break
    cnt_page = 0
                            
    a += 1 
    b = str(a)

    try :
        driver.find_element(By.LINK_TEXT ,'%s' %b).click() 
    except :
        driver.find_element(By.XPATH,'//*[@id="content"]/div[2]/div[5]/div/a[3]').click()
        
print("요청하신 작업이 모두 완료되었습니다")

# Step 10. 수집된 데이터를 xls와 csv 형태로 저장하기
# 현재 날짜와 시간으로 폴더 만들고 파일 이름 설정하기
import os

n = time.localtime()
s = '%04d-%02d-%02d-%02d-%02d-%02d' %(n.tm_year, n.tm_mon, n.tm_mday, n.tm_hour, n.tm_min, n.tm_sec)

os.makedirs(f_dir+'서울시 응답소'+'-'+s+'-'+'상담내역')

fc_name = f_dir+'서울시 응답소'+'-'+s+'-'+'상담내역'+'\\'+'서울시 응답소'+'-'+s+'-'+'상담내역'+'.csv'
fx_name = f_dir+'서울시 응답소'+'-'+s+'-'+'상담내역'+'\\'+'서울시 응답소'+'-'+s+'-'+'상담내역'+'.xlsx'
ft_name = f_dir+'서울시 응답소'+'-'+s+'-'+'상담내역'+'\\'+'서울시 응답소'+'-'+s+'-'+'상담내역'+'.txt'
# 데이터 프레임 생성 후 xls , csv 형식으로 저장하기
import pandas as pd 

df = pd.DataFrame()
df['번호']=no2
df['제목']=pd.Series(title2)
df['공개일']=pd.Series(contents2)
df['상담내용']=pd.Series(contents3)

# xls 형태로 저장하기
df.to_excel(fx_name,index=False, engine='openpyxl')

# csv 형태로 저장하기
df.to_csv(fc_name,index=False, encoding="utf-8-sig")

print('요청하신 데이터 수집 작업이 정상적으로 완료되었습니다')

연습문제 : 서울시 응답소 게시판 크롤링하기 
크롤링 할 총 페이지 번호: 6




1 페이지 내용 수집 시작합니다 =======================


1 번째 게시글의 상세내역을 추출하고 있습니다============
1.제목 : 광화문 625참전기념구조물설치 건
2.공개일 : 2025-02-06
3.상담내용 : 광화문은 625참전국 기념구조물은 어울리지 않음전쟁기념관과 유엔참전용사 묘지 등으로 충분하다 사료됨.


2 번째 게시글의 상세내역을 추출하고 있습니다============
1.제목 : 경희궁 완전복원
2.공개일 : 2025-02-05
3.상담내용 : 경희궁은 조선의 귀중한 궁궐입니다! 반드시 복원해야합니다! 권역을 침범한 시설물들을 철거하고 조속히 완전 복원해주세요!


3 번째 게시글의 상세내역을 추출하고 있습니다============
1.제목 : 서울특별시장께 서대문구 홍제천 제방공사 관련 나무 보존 및 법적 문제 문의
2.공개일 : 2025-02-05
3.상담내용 : 서울특별시장 귀하제목: 서대문구 홍제천 제방공사 관련 나무 보존 및 법적 문제 문의서론2024년 11월 완료된 서대문구 홍제천 백련교 앞 제방공사 및 그 이전에 공사한 제방공사들로 인해 수십 년 된 나무들이 모두 베어져 없어진 상황과 현재 아직 남아 있기는 하나 앞으로 사라질 나무들에 관해 문의드립니다. 첨부한 사진을 통해 제방공사가 완료된 곳은 깔끔하기는 하나 나무가 없어 그늘이 없고 황량한 모습을 보여주고 있으나, 반면 마포구에 있는 홍제천은 서대문구에 있는 제방처럼 깔끔하지는 않으나 나무와 제방이 어우러져 자연적인 멋을 유지하고 있습니다. 이런 각기 다른 구청의 제방과 관련하여 서울시의 입장 및 정책을 문의드리고자 합니다.본론1. - 서대문구 홍제천 제방공사가 완료된 후, 수십 년 된 나무들이 모두 베어져 없어진 것을 알게 되었습니다. 나무들과 다른 식물들을 모두 제거해 버린 것으로 인해 깔끔해 보일지는 모르나 그늘은 없어지고 황량한 모습이 되었습니다. 이 제방

In [1]:
#Step 1. 필요한 모듈을 로딩합니다
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
import time 
from selenium.webdriver.chrome.options import Options

chrome_options = Options()
chrome_options.add_argument("--force-device-scale-factor=1")

#Step 2. 사용자에게 검색 관련 정보들을 입력 받습니다.
print("=" *100)
print(" 이 크롤러는 대한민국 구석구석 사이트 정보 수집용 웹크롤러입니다.")
print("=" *100)
query_txt = input('1.수집할 자료의 키워드는 무엇입니까?: ')

#Step 3. 수집된 데이터를 저장할 파일 이름 입력받기 
f_dir = input("2.파일을 저장할 폴더명만 쓰세요(기본값:c:\\py_temp\\):")
if f_dir == '' :
    f_dir="c:\\py_temp\\"

#Step 4. 크롬 드라이버 설정 및 웹 페이지 열기
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s, options=chrome_options)

url = 'http://korean.visitkorea.or.kr'
driver.get(url)
driver.maximize_window()
time.sleep(2)

#Step 5. 자동으로 검색어 입력 후 조회하기
element = driver.find_element(By.XPATH,'//*[@id="inp_search"]')
driver.find_element(By.XPATH,'//*[@id="inp_search"]').click()
element.send_keys(query_txt)
driver.find_element(By.XPATH,'//*[@id="placeHolder"]/a').click()


#Step 7.Beautiful Soup 로 본문 내용만 추출하기
from bs4 import BeautifulSoup
html_1 = driver.page_source
soup_1 = BeautifulSoup(html_1, 'html.parser')

#Step 8. 총 검색 건수를 보여주고 수집할 건수 입력받기
import math
total_cnt = soup_1.find('div','searchBox pd').find('span','num').get_text()
print('검색하신 키워드 %s (으)로 총 %s 건의 학위논문이 검색되었습니다' %(query_txt,total_cnt))
cnt = int(input('이 중에서 몇 건을 수집하시겠습니까?: '))
page_cnt = math.ceil(cnt / 10)
print('%s 건의 데이터를 수집하기 위해 %s 페이지의 게시물을 조회합니다.' %(cnt,page_cnt))
print("\n")

#Step 9. 데이터 수집하기
no2=[]           # 게시글 번호 컬럼
title2=[ ]       # 게시글 제목 컬럼
author2=[]       # 논문 저자 컬럼
company2=[ ]     # 소속 기관 컬럼
date2=[ ]        # 게시글 날짜 컬럼
suksa2=[ ]       # 국내석사 컬럼
contents2=[]     # 초록내용
full_url2=[]     # 논문 원본 URL

no = 1           # 게시글 번호 초기값
            
for a in range(1,page_cnt+1) :
    print("\n")
    print("%s 페이지 내용 수집 시작합니다 =======================" %a)

    time.sleep(2)
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    content_list = soup.find('div','srchResultListW').find_all('li')

    for i in content_list:
        # 논문 제목 체크하기
        try:
            title=i.find('p','title').get_text().strip()
        except :
            continue 
        else :
            # 1.게시글 번호
            print("\n")
            print("%s 번째 정보를 추출하고 있습니다============" %no)
            no2.append(no)
            print("1.번호 : %s" %no)
            
            # 2. 논문 제목
            title2.append(title.strip())
            print("2.제목 : %s" %title.strip())

            # 3. 작성자
            try :
                author=i.find('p','etc').find('span','writer').get_text().strip()
            except :
                author = '작성자가 없습니다'
                print("3.작성자 : %s" %author.strip())
                author2.append(author.strip())
            else :
                author2.append(author.strip())
                print("3.작성자 : %s" %author.strip())

            # 4. 소속기관
            try :
                company=i.find('p','etc').find('span','assigned').get_text().strip()
            except :
                company='소속 기관이 없습니다'
                company2.append(company.strip())
                print("4.소속기관 : %s" %company.strip())
            else :
                company2.append(company.strip())
                print("4.소속기관 : %s" %company.strip())

            # 5. 발표날짜
            try :
                date_1 =i.find('p','etc').find_all('span')
                date_2 = date_1[2].get_text().strip()
            except :
                date_2='발표날짜가 없습니다'
                date2.append(date_2)
                print("5.발표년도 : %s" %date_2)
            else :
                date2.append(date_2)
                print("5.발표년도 : %s" %date_2)

            # 6.학위여부
            try :
                suksa_1 =i.find('p','etc').find_all('span')
                suksa_2 = suksa_1[3].get_text().strip()
            except :
                suksa_2='학위가 없습니다'
                suksa2.append(suksa_2)
                print("6.학위여부 : %s" %suksa_2)
            else :
                suksa2.append(suksa_2)
                print("6.학위여부 : %s" %suksa_2)

            # 7.초록 내용-해당 논문의 상세 내역에서 추출할 수 있음.    
            url_1 = i.find('p','title').find('a')['href']
            full_url = 'http://www.riss.kr'+url_1
            time.sleep(1)
            driver.get(full_url)

            html_1 = driver.page_source
            soup_1 = BeautifulSoup(html_1, 'html.parser')  
            try :
                cont=soup_1.find("div","text").find('p').get_text().replace("\n","").strip()
            except :
                cont='초록이 없습니다'
                contents2.append(cont)
                print("7.초록내용 : %s" %cont)
            else :
                contents2.append(cont)
                print("7.초록내용 : %s" %cont)

            time.sleep(1)

            # 8.논문 url 주소
            full_url2.append(full_url)
            print('8.논문 URL 주소:' , full_url)

            driver.back()  # 이전 페이지로 돌아가기

            time.sleep(2)

            no += 1
            
            if no > cnt :
                break 
                            
    a += 1 
    b = str(a)

    try :
        driver.find_element(By.LINK_TEXT ,'%s' %b).click() 
    except :
        driver.find_element(By.LINK_TEXT,'다음 페이지로').click()
        
print("요청하신 작업이 모두 완료되었습니다")

# Step 10. 수집된 데이터를 xls와 csv 형태로 저장하기
# 현재 날짜와 시간으로 폴더 만들고 파일 이름 설정하기
import os

n = time.localtime()
s = '%04d-%02d-%02d-%02d-%02d-%02d' %(n.tm_year, n.tm_mon, n.tm_mday, n.tm_hour, n.tm_min, n.tm_sec)

os.makedirs(f_dir+'RISS'+'-'+s+'-'+'학위논문')

fc_name = f_dir+'RISS'+'-'+s+'-'+'학위논문'+'\\'+'RISS'+'-'+s+'-'+'학위논문'+'.csv'
fx_name = f_dir+'RISS'+'-'+s+'-'+'학위논문'+'\\'+'RISS'+'-'+s+'-'+'학위논문'+'.xls'

# 데이터 프레임 생성 후 xls , csv 형식으로 저장하기
import pandas as pd 

df = pd.DataFrame()
df['번호']=no2
df['제목']=pd.Series(title2)
df['저자']=pd.Series(author2)
df['소속(발행)기관']=pd.Series(company2)
df['날짜']=pd.Series(date2)
df['학위(논문일경우)']=pd.Series(suksa2)
df['초록(논문일경우)']=pd.Series(contents2)
df['자료URL주소']=pd.Series(full_url2)

# xls 형태로 저장하기
df.to_excel(fx_name,index=False, engine='openpyxl')

# csv 형태로 저장하기
df.to_csv(fc_name,index=False, encoding="utf-8-sig")

print('요청하신 데이터 수집 작업이 정상적으로 완료되었습니다')

 이 크롤러는 대한민국 구석구석 사이트 정보 수집용 웹크롤러입니다.


AttributeError: 'NoneType' object has no attribute 'find'